# Wireframe de mano con Intel RealSense + MediaPipe

Detecta **una mano derecha** en tiempo real y dibuja su esqueleto (21 landmarks y conexiones) sobre el video de la cámara RealSense.

**Flujo**
1. RealSense entrega frames de color (640x480 @ 30 fps, BGR).
2. Se convierte a RGB (MediaPipe lo requiere).
3. `mp.solutions.hands` extrae los 21 landmarks y la lateralidad (`Left`/`Right`).
4. Si la mano es `Right`, se dibuja el wireframe y el texto *Mano Derecha Detectada*.
5. Se muestra en una ventana OpenCV. Tecla **`q`** para salir.

**Notas**
- Por ahora solo color (sin depth).
- MediaPipe asume imagen tipo espejo (selfie). Como la RealSense no refleja la imagen, la etiqueta `Right` puede corresponder a la mano **izquierda** real. Como sucede esto, se cambia el filtro a `"Left"`.
---

## Entorno (Jetson Orin Nano, Ubuntu 22.04, Python 3.10)

```bash
python3.10 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements-22.04.txt
```

- No instalar `opencv-python` aparte (mediapipe ya trae `opencv-contrib-python`; ambos juntos chocan).
- `protobuf` debe ser 4.25.x y `numpy<2`.
- Si `pyrealsense2` no ve la cámara: instalar `librealsense` / reglas udev y revisar con `rs-enumerate-devices`.
- La ventana de OpenCV necesita sesión gráfica (monitor o `DISPLAY` activo).
- Si la ventana se cierra con la X en vez de `q`, reiniciar el kernel.
---

## 1. Imports y configuración de MediaPipe

In [1]:
import warnings
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,          # Buscar máximo 1 mano en pantalla
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 2. Captura RealSense y bucle de detección
Ejecutar la celda; presionar `q` sobre la ventana de video para terminar.

In [2]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30) #Por ahora solo con color

print("Iniciando cámara... Presiona la tecla 'q' sobre la ventana de video para salir.")
pipeline.start(config)

try:
    while True:
        # Esperar y extraer el fotograma
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()

        if not color_frame:
            continue

        color_image = np.asanyarray(color_frame.get_data())

        # Conversion de BGR a RGB para el MediaPipe
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)

        # Extraer los 21 puntos clave (Landmarks)
        results = hands.process(image_rgb)

        # Dibujar el wireframe si se detecta una mano
        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):

                # Filtrar para trabajar únicamente con la mano derecha
                label = handedness.classification[0].label
                if label == "Left":
                    # Trazar el esqueleto y las conexiones
                    mp_drawing.draw_landmarks(
                        color_image,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS
                    )
                    cv2.putText(color_image, "Mano Derecha Detectada", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    
        # Mostrar el resultado en una ventana interactiva de OpenCV
        cv2.imshow('Wireframe Mano RealSense', color_image)

        # Romper el ciclo si el usuario presiona 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    print("Pipeline detenido y ventanas cerradas.")

Iniciando cámara... Presiona la tecla 'q' sobre la ventana de video para salir.


W0000 00:00:1792523034.106611   16178 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792523034.131719   16179 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/udi/miniforge3/envs/RealSense/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Pipeline detenido y ventanas cerradas.


## Troubleshooting

| Error | Causa / solución |
|---|---|
| `RuntimeError: No device connected` | RealSense no conectada o sin permisos (udev). |
| `module 'mediapipe' has no attribute 'solutions'` / `No module named 'mediapipe.python'` | Instalación mezclada. Recrear el venv con `requirements-22.04.txt`. |
| `libopencv_*.so.412: cannot open shared object file` | Wheel de mediapipe compilado contra otro OpenCV. Usar las versiones del requirements, no wheels de terceros. |
| Kernel muere al importar | Libs nativas incompatibles o poca RAM; probar el script en terminal. |